In [1]:
import chromadb
import os
import json
from chromadb.utils import embedding_functions
from datetime import datetime
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [2]:
chroma_client = chromadb.Client()

**OpenAI embedding models**

| Model                     | Dimensions | Primary Use Case                                | Notes                                              |
|---------------------------|------------|-------------------------------------------------|----------------------------------------------------|
| text-embedding-3-small    | 1,536      | Cost-effective semantic search & multilingual   | ~2× cheaper than 3-large; good across ≥50 languages |
| text-embedding-3-large    | 3,072      | High-precision retrieval (e.g., legal, medical) | Highest accuracy; ~2× cost of 3-small              |
| text-embedding-ada-002    | 1,536      | General-purpose embeddings, legacy compatibility | Being superseded by 3-series models                |


In [3]:
# By default, Chroma uses the Sentence Transformers all-MiniLM-L6-v2 model to create embeddings
default_ef = embedding_functions.DefaultEmbeddingFunction()

# You can also use a custom embedding function, for example, using OpenAI's text-embedding-ada-002 model
openai_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key = os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-large"
)


In [4]:
# delete if it exists
if "my_collection" in chroma_client.list_collections():
    chroma_client.delete_collection(name="my_collection")

# Create a new collection in ChromaDB
collection = chroma_client.get_or_create_collection(
    name="my_collection",
    metadata={
        "description": "my first Chroma collection",
        "created": str(datetime.now())
    },
    embedding_function=openai_fn,
)

# insert some documents into the collection
collection.upsert(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges",
        "This is a document about cherries"
    ],
    ids=["id1", "id2", "id3"],
)

In [11]:
results = collection.query(
    query_texts=["This is a query document about switzerland"], # Chroma will embed this
    n_results=2 # how many results to return
)
print(json.dumps(results, indent=2))

{
  "ids": [
    [
      "id3",
      "id2"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "This is a document about cherries",
      "This is a document about oranges"
    ]
  ],
  "uris": null,
  "data": null,
  "metadatas": [
    [
      null,
      null
    ]
  ],
  "distances": [
    [
      1.2900141477584839,
      1.3147884607315063
    ]
  ],
  "included": [
    "distances",
    "documents",
    "metadatas"
  ]
}


In [6]:
# list all collections
print(chroma_client.list_collections())

# read the collection metadata
print("Collection metadata:", json.dumps(collection.metadata, indent=2))

['my_collection']
Collection metadata: {
  "description": "my first Chroma collection",
  "created": "2025-05-31 14:51:34.552343"
}


In [7]:
# read all documents in the collection
results2 = collection.get(
    ids=["id1", "id2", "id3"],
    include=["embeddings", "documents", "metadatas"] # include embeddings, documents, and metadata in the result
)
print(results2)

{'ids': ['id1', 'id2', 'id3'], 'embeddings': array([[-0.03052354,  0.00046051, -0.0160323 , ..., -0.00129042,
        -0.00323098,  0.00744263],
       [-0.04782376, -0.00576385, -0.01699002, ...,  0.00204073,
         0.00136667, -0.00337353],
       [-0.00329518, -0.0213238 , -0.01758006, ...,  0.00075856,
        -0.00092461, -0.01049801]], shape=(3, 3072)), 'documents': ['This is a document about pineapple', 'This is a document about oranges', 'This is a document about cherries'], 'uris': None, 'data': None, 'metadatas': [None, None, None], 'included': [<IncludeEnum.embeddings: 'embeddings'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


In [8]:
# get the embeddings
embeddings = results2['embeddings']
print(embeddings.shape)
embeddings


(3, 3072)


array([[-0.03052354,  0.00046051, -0.0160323 , ..., -0.00129042,
        -0.00323098,  0.00744263],
       [-0.04782376, -0.00576385, -0.01699002, ...,  0.00204073,
         0.00136667, -0.00337353],
       [-0.00329518, -0.0213238 , -0.01758006, ...,  0.00075856,
        -0.00092461, -0.01049801]], shape=(3, 3072))